In [0]:
-- =============================================================================
-- NOTEBOOK: 03_build_gold_layer
-- PURPOSE:  Build Gold fact tables and pre-computed trend views
--           from Silver dimension tables
-- CATALOG:  people_analytics
-- SCHEMA:   gold
-- AUTHOR:   Portfolio Project — Adobe People Analytics
-- DATE:     2026-04-09
--
-- PERFORMANCE DESIGN:
--   All time comparisons (MoM, YoY, rolling averages) are pre-computed
--   in SQL using window functions at refresh time — NOT in DAX at query time.
--   Power BI reads pre-aggregated columns. Line charts never fire live
--   calculations against row-level fact tables.
-- =============================================================================

-- CELL 1 — Set catalog and schema
USE CATALOG people_analytics;
USE SCHEMA gold;

SELECT 'Gold layer build started' AS status, current_timestamp() AS run_ts;

In [0]:
-- =============================================================================
-- CELL 2 — fact_headcount_snapshot
-- One row per employee per month-end snapshot
-- This is the primary fact table for headcount, tenure, and payroll metrics
--
-- GRAIN: employee × month
-- ADDITIVE: SUM(is_active) = headcount at any point. Fully additive.
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.gold.fact_headcount_snapshot
COMMENT 'Monthly headcount snapshot. One row per employee per month.
SUM(is_active) = active headcount. Fully additive — safe to SUM across any dimension.'
PARTITIONED BY (snapshot_year, snapshot_month)
AS
WITH

-- Generate month-end spine for the data range
month_spine AS (
    SELECT DISTINCT
        DATE_TRUNC('MONTH', date) AS month_start,
        LAST_DAY(date)            AS month_end,
        year                      AS snapshot_year,
        month                     AS snapshot_month,
        yyyymm                    AS snapshot_yyyymm
    FROM people_analytics.silver.dim_date
    WHERE date = LAST_DAY(date)          -- month-end dates only
      AND year BETWEEN 2023 AND 2026
      AND date <= CURRENT_DATE()
),

-- Current employee versions only for joining
current_employees AS (
    SELECT *
    FROM people_analytics.silver.dim_employee
    WHERE is_current = 1
)

SELECT
    -- Keys
    CAST(ROW_NUMBER() OVER (
        ORDER BY e.emp_id, ms.month_end
    ) AS BIGINT)                                AS snapshot_sk,
    snapshot_year * 10000 + snapshot_month * 100 + 1 AS date_sk,
    e.employee_sk                               AS employee_sk,
    e.dept_id                                   AS dept_id,
    e.job_code                                  AS job_code,
    e.comp_band                                 AS comp_band,

    -- Snapshot dimensions
    ms.month_start                              AS snapshot_month_start,
    ms.month_end                                AS snapshot_month_end,
    ms.snapshot_year                            AS snapshot_year,
    ms.snapshot_month                           AS snapshot_month,
    ms.snapshot_yyyymm                          AS snapshot_yyyymm,

    -- Employee at snapshot time
    e.emp_id                                    AS emp_id,
    e.full_name                                 AS full_name,
    e.gender                                    AS gender,
    e.ethnicity                                 AS ethnicity,
    e.employment_type                           AS employment_type,
    e.dept_name                                 AS dept_name,
    e.business_unit                             AS business_unit,
    e.region                                    AS region,
    e.job_title                                 AS job_title,
    e.job_family                                AS job_family,
    e.job_level                                 AS job_level,
    e.is_people_manager                         AS is_people_manager,
    e.salary                                    AS salary,
    e.compa_ratio                               AS compa_ratio,
    e.status                                    AS status,

    -- Additive headcount flags
    CASE WHEN e.status = 'Active'
         AND e.hire_date <= ms.month_end
         AND (e.termination_date IS NULL
              OR e.termination_date > ms.month_end)
         THEN 1 ELSE 0 END                      AS is_active,

    CASE WHEN e.hire_date BETWEEN ms.month_start
                              AND ms.month_end
         THEN 1 ELSE 0 END                      AS is_new_hire,

    CASE WHEN e.termination_date BETWEEN ms.month_start
                                     AND ms.month_end
         THEN 1 ELSE 0 END                      AS is_termination,

    CASE WHEN e.termination_type = 'Voluntary'
          AND e.termination_date BETWEEN ms.month_start
                                     AND ms.month_end
         THEN 1 ELSE 0 END                      AS is_voluntary_exit,

    -- Tenure at snapshot
    DATEDIFF(ms.month_end, e.hire_date)         AS tenure_days,
    ROUND(DATEDIFF(ms.month_end, e.hire_date)
          / 365.25, 2)                          AS tenure_years,

    -- Tenure band at snapshot
    CASE
        WHEN DATEDIFF(ms.month_end, e.hire_date) / 365.25 < 1   THEN '< 1 Year'
        WHEN DATEDIFF(ms.month_end, e.hire_date) / 365.25 < 3   THEN '1-3 Years'
        WHEN DATEDIFF(ms.month_end, e.hire_date) / 365.25 < 5   THEN '3-5 Years'
        WHEN DATEDIFF(ms.month_end, e.hire_date) / 365.25 < 10  THEN '5-10 Years'
        ELSE '10+ Years'
    END                                         AS tenure_band,

    -- Audit
    current_timestamp()                         AS load_ts

FROM current_employees e
CROSS JOIN month_spine ms
WHERE e.hire_date <= ms.month_end;

SELECT 'fact_headcount_snapshot rows:' AS table_name,
       COUNT(*) AS row_count
FROM people_analytics.gold.fact_headcount_snapshot;


In [0]:
-- =============================================================================
-- CELL 3 — fact_attrition_event
-- One row per employee exit event
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.gold.fact_attrition_event
COMMENT 'One row per employee termination event.
Join to dim_employee on employee_sk for employee attributes at time of exit.'
AS
WITH last_perf AS (
    SELECT
        emp_id,
        MAX(rating) AS last_rating
    FROM people_analytics.bronze.performance_raw
    GROUP BY emp_id
)
SELECT
    CAST(ROW_NUMBER() OVER (
        ORDER BY e.emp_id
    ) AS BIGINT)                                            AS attrition_sk,

    -- Date key
    CAST(DATE_FORMAT(e.termination_date, 'yyyyMMdd') AS INT) AS date_sk,

    -- Employee
    e.employee_sk                                           AS employee_sk,
    e.emp_id                                                AS emp_id,
    e.full_name                                             AS full_name,
    e.dept_id                                               AS dept_id,
    e.dept_name                                             AS dept_name,
    e.business_unit                                         AS business_unit,
    e.job_level                                             AS job_level,
    e.job_family                                            AS job_family,
    e.gender                                                AS gender,
    e.ethnicity                                             AS ethnicity,

    -- Exit details
    e.termination_date                                      AS termination_date,
    e.termination_reason                                    AS termination_reason,
    e.termination_type                                      AS termination_type,
    YEAR(e.termination_date)                                AS exit_year,
    MONTH(e.termination_date)                               AS exit_month,

    -- Tenure at exit
    DATEDIFF(e.termination_date, e.hire_date)               AS tenure_at_exit_days,
    ROUND(DATEDIFF(e.termination_date, e.hire_date)
          / 365.25, 2)                                      AS tenure_at_exit_years,

    -- Exit flags
    CASE WHEN DATEDIFF(e.termination_date, e.hire_date) < 90
         THEN 1 ELSE 0 END                                  AS is_90_day_exit,

    CASE WHEN COALESCE(p.last_rating, 0) >= 4
         THEN 1 ELSE 0 END                                  AS was_high_performer,

    CASE WHEN e.termination_type = 'Voluntary'
         AND COALESCE(p.last_rating, 0) >= 4
         THEN 1 ELSE 0 END                                  AS is_regrettable,

    -- Salary at exit
    e.salary                                                AS salary_at_exit,
    e.compa_ratio                                           AS compa_ratio_at_exit,

    current_timestamp()                                     AS load_ts

FROM people_analytics.silver.dim_employee e
LEFT JOIN last_perf p ON e.emp_id = p.emp_id
WHERE e.is_current  = 1
  AND e.status      = 'Terminated'
  AND e.termination_date IS NOT NULL;

SELECT 'fact_attrition_event rows:' AS table_name,
       COUNT(*) AS row_count
FROM people_analytics.gold.fact_attrition_event;


In [0]:
-- =============================================================================
-- CELL 4 — fact_compensation_history
-- One row per compensation change event
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.gold.fact_compensation_history
COMMENT 'One row per compensation change. Includes compa-ratio and range penetration.'
AS
SELECT
    CAST(ROW_NUMBER() OVER (
        ORDER BY c.emp_id, c.effective_date
    ) AS BIGINT)                                                AS comp_sk,

    CAST(DATE_FORMAT(CAST(c.effective_date AS DATE),
         'yyyyMMdd') AS INT)                                    AS date_sk,

    e.employee_sk                                               AS employee_sk,
    c.emp_id                                                    AS emp_id,
    CAST(c.effective_date AS DATE)                              AS effective_date,
    YEAR(CAST(c.effective_date AS DATE))                        AS effective_year,
    MONTH(CAST(c.effective_date AS DATE))                       AS effective_month,

    -- Salary change
    CAST(c.old_salary         AS DECIMAL(18,2))                 AS old_salary,
    CAST(c.new_salary         AS DECIMAL(18,2))                 AS new_salary,
    CAST(c.change_amount      AS DECIMAL(18,2))                 AS change_amount,
    CAST(c.change_pct         AS DECIMAL(8,2))                  AS change_pct,
    c.change_reason                                             AS change_reason,

    -- Band metrics
    c.comp_band                                                 AS comp_band,
    CAST(c.compa_ratio        AS DECIMAL(8,3))                  AS compa_ratio,
    CAST(c.range_penetration  AS DECIMAL(8,3))                  AS range_penetration,

    -- Employee context at time of change
    e.dept_name                                                 AS dept_name,
    e.business_unit                                             AS business_unit,
    e.job_level                                                 AS job_level,
    e.job_family                                                AS job_family,
    e.gender                                                    AS gender,

    current_timestamp()                                         AS load_ts

FROM people_analytics.bronze.comp_changes_raw c
LEFT JOIN people_analytics.silver.dim_employee e
    ON c.emp_id = e.emp_id
   AND e.is_current = 1;

SELECT 'fact_compensation_history rows:' AS table_name,
       COUNT(*) AS row_count
FROM people_analytics.gold.fact_compensation_history;



In [0]:
-- =============================================================================
-- CELL 5 — fact_performance_review
-- One row per performance review
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.gold.fact_performance_review
COMMENT 'One row per performance review cycle per employee.'
AS
SELECT
    CAST(ROW_NUMBER() OVER (
        ORDER BY p.emp_id, p.review_cycle
    ) AS BIGINT)                                                AS review_sk,

    CAST(DATE_FORMAT(CAST(p.review_date AS DATE),
         'yyyyMMdd') AS INT)                                    AS date_sk,

    e.employee_sk                                               AS employee_sk,
    p.emp_id                                                    AS emp_id,
    p.review_cycle                                              AS review_cycle,
    CAST(p.review_date AS DATE)                                 AS review_date,
    CAST(p.rating      AS INT)                                  AS rating,
    p.rating_label                                              AS rating_label,
    CAST(p.is_high_performer AS INT)                            AS is_high_performer,
    CAST(p.is_pip            AS INT)                            AS is_pip,

    -- Employee context
    e.dept_name                                                 AS dept_name,
    e.business_unit                                             AS business_unit,
    e.job_level                                                 AS job_level,
    e.job_family                                                AS job_family,
    e.gender                                                    AS gender,
    e.manager_id                                                AS manager_id,

    current_timestamp()                                         AS load_ts

FROM people_analytics.bronze.performance_raw p
LEFT JOIN people_analytics.silver.dim_employee e
    ON p.emp_id = e.emp_id
   AND e.is_current = 1;

SELECT 'fact_performance_review rows:' AS table_name,
       COUNT(*) AS row_count
FROM people_analytics.gold.fact_performance_review;


In [0]:
-- =============================================================================
-- CELL 6 — v_headcount_trend
-- Pre-computed monthly headcount with MoM and YoY comparisons
-- Power BI line charts read this view — no DAX time intelligence at render time
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.gold.v_headcount_trend
COMMENT 'Pre-computed monthly headcount trend with MoM and YoY comparisons.
Power BI line charts read columns directly — no DAX TI needed at render time.
Performance: computed once at refresh, not per visual render.'
AS
WITH monthly AS (
    SELECT
        snapshot_yyyymm,
        snapshot_year,
        snapshot_month,
        snapshot_month_start,
        snapshot_month_end,
        SUM(is_active)         AS active_headcount,
        SUM(is_new_hire)       AS new_hires,
        SUM(is_termination)    AS total_exits,
        SUM(is_voluntary_exit) AS voluntary_exits,
        ROUND(AVG(CASE WHEN is_active = 1
                       THEN salary END), 0)     AS avg_salary,
        ROUND(AVG(CASE WHEN is_active = 1
                       THEN tenure_years END), 2) AS avg_tenure_years
    FROM people_analytics.gold.fact_headcount_snapshot
    GROUP BY snapshot_yyyymm, snapshot_year, snapshot_month,
             snapshot_month_start, snapshot_month_end
)
SELECT
    snapshot_yyyymm,
    snapshot_year,
    snapshot_month,
    snapshot_month_start,
    snapshot_month_end,
    active_headcount,
    new_hires,
    total_exits,
    voluntary_exits,
    active_headcount - total_exits + new_hires  AS net_change,
    avg_salary,
    avg_tenure_years,

    -- MoM comparisons (pre-computed — no DAX needed)
    LAG(active_headcount, 1) OVER (ORDER BY snapshot_yyyymm) AS prior_month_hc,
    active_headcount -
        LAG(active_headcount, 1) OVER (ORDER BY snapshot_yyyymm) AS mom_hc_delta,
    ROUND(100.0 * (active_headcount -
        LAG(active_headcount, 1) OVER (ORDER BY snapshot_yyyymm)) /
        NULLIF(LAG(active_headcount, 1) OVER (ORDER BY snapshot_yyyymm), 0),
    2)                                          AS mom_hc_pct,

    -- YoY comparisons (pre-computed — no DAX needed)
    LAG(active_headcount, 12) OVER (ORDER BY snapshot_yyyymm) AS prior_year_hc,
    active_headcount -
        LAG(active_headcount, 12) OVER (ORDER BY snapshot_yyyymm) AS yoy_hc_delta,
    ROUND(100.0 * (active_headcount -
        LAG(active_headcount, 12) OVER (ORDER BY snapshot_yyyymm)) /
        NULLIF(LAG(active_headcount, 12) OVER (ORDER BY snapshot_yyyymm), 0),
    2)                                          AS yoy_hc_pct,

    -- Rolling averages (pre-computed)
    ROUND(AVG(active_headcount) OVER (
        ORDER BY snapshot_yyyymm
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 0)                                       AS rolling_3m_avg_hc,

    ROUND(AVG(active_headcount) OVER (
        ORDER BY snapshot_yyyymm
        ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
    ), 0)                                       AS rolling_12m_avg_hc,

    -- Cumulative joiners waterfall
    SUM(new_hires) OVER (
        ORDER BY snapshot_yyyymm
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    )                                           AS cumulative_joiners,

    current_timestamp()                         AS load_ts

FROM monthly
ORDER BY snapshot_yyyymm;

SELECT 'v_headcount_trend rows:' AS table_name,
       COUNT(*) AS row_count
FROM people_analytics.gold.v_headcount_trend;

In [0]:
-- =============================================================================
-- CELL 7 — v_attrition_trend
-- Pre-computed monthly attrition rate with rolling average
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.gold.v_attrition_trend
COMMENT 'Pre-computed monthly attrition trend. Rolling averages pre-computed.
Power BI attrition line charts read columns — no DAX TI at render time.'
AS
WITH monthly AS (
    SELECT
        h.snapshot_yyyymm,
        h.snapshot_year,
        h.snapshot_month,
        h.snapshot_month_start,
        SUM(h.is_active)                                AS active_hc,
        SUM(h.is_termination)                           AS total_exits,
        SUM(h.is_voluntary_exit)                        AS voluntary_exits,
        SUM(CASE WHEN h.is_termination = 1
                  AND h.status = 'Terminated'
                  AND ae.is_regrettable = 1
                 THEN 1 ELSE 0 END)                     AS regrettable_exits
    FROM people_analytics.gold.fact_headcount_snapshot h
    LEFT JOIN people_analytics.gold.fact_attrition_event ae
        ON h.emp_id = ae.emp_id
    GROUP BY h.snapshot_yyyymm, h.snapshot_year,
             h.snapshot_month, h.snapshot_month_start
)
SELECT
    snapshot_yyyymm,
    snapshot_year,
    snapshot_month,
    snapshot_month_start,
    active_hc,
    total_exits,
    voluntary_exits,
    regrettable_exits,

    -- Monthly attrition rates
    ROUND(100.0 * voluntary_exits /
          NULLIF(active_hc, 0), 2)              AS voluntary_attrition_rate,
    ROUND(100.0 * total_exits /
          NULLIF(active_hc, 0), 2)              AS total_attrition_rate,
    ROUND(100.0 * regrettable_exits /
          NULLIF(active_hc, 0), 2)              AS regrettable_attrition_rate,

    -- Rolling 3-month average (smoothed trend line)
    ROUND(AVG(100.0 * voluntary_exits /
              NULLIF(active_hc, 0)) OVER (
        ORDER BY snapshot_yyyymm
        ROWS BETWEEN 2 PRECEDING AND CURRENT ROW
    ), 2)                                       AS rolling_3m_attrition,

    -- Rolling 12-month attrition
    ROUND(AVG(100.0 * voluntary_exits /
              NULLIF(active_hc, 0)) OVER (
        ORDER BY snapshot_yyyymm
        ROWS BETWEEN 11 PRECEDING AND CURRENT ROW
    ), 2)                                       AS rolling_12m_attrition,

    -- YoY attrition comparison
    LAG(ROUND(100.0 * voluntary_exits /
              NULLIF(active_hc, 0), 2), 12)
        OVER (ORDER BY snapshot_yyyymm)         AS prior_year_attrition_rate,

    current_timestamp()                         AS load_ts

FROM monthly
ORDER BY snapshot_yyyymm;

SELECT 'v_attrition_trend rows:' AS table_name,
       COUNT(*) AS row_count
FROM people_analytics.gold.v_attrition_trend;

In [0]:
-- =============================================================================
-- CELL 8 — v_compensation_trend
-- Pre-computed monthly compensation trend by job band
-- =============================================================================

CREATE OR REPLACE TABLE people_analytics.gold.v_compensation_trend
COMMENT 'Pre-computed monthly salary trends by job band.
Feeds compensation line charts in Power BI.'
AS
WITH monthly_comp AS (
    SELECT
        snapshot_yyyymm,
        snapshot_year,
        snapshot_month,
        snapshot_month_start,
        comp_band,
        SUM(is_active)                          AS headcount,
        ROUND(AVG(CASE WHEN is_active = 1
                       THEN salary END), 0)     AS avg_salary,
        ROUND(AVG(CASE WHEN is_active = 1
                       THEN compa_ratio END), 3) AS avg_compa_ratio,
        MIN(CASE WHEN is_active = 1
                 THEN salary END)               AS min_salary,
        MAX(CASE WHEN is_active = 1
                 THEN salary END)               AS max_salary
    FROM people_analytics.gold.fact_headcount_snapshot
    WHERE comp_band IS NOT NULL
    GROUP BY snapshot_yyyymm, snapshot_year, snapshot_month,
             snapshot_month_start, comp_band
)
SELECT
    snapshot_yyyymm,
    snapshot_year,
    snapshot_month,
    snapshot_month_start,
    comp_band,
    headcount,
    avg_salary,
    avg_compa_ratio,
    min_salary,
    max_salary,

    -- MoM salary change per band
    LAG(avg_salary, 1) OVER (
        PARTITION BY comp_band
        ORDER BY snapshot_yyyymm
    )                                           AS prior_month_avg_salary,

    ROUND(avg_salary -
        LAG(avg_salary, 1) OVER (
            PARTITION BY comp_band
            ORDER BY snapshot_yyyymm
        ), 0)                                   AS mom_salary_delta,

    -- YoY salary change per band
    LAG(avg_salary, 12) OVER (
        PARTITION BY comp_band
        ORDER BY snapshot_yyyymm
    )                                           AS prior_year_avg_salary,

    ROUND(100.0 * (avg_salary -
        LAG(avg_salary, 12) OVER (
            PARTITION BY comp_band
            ORDER BY snapshot_yyyymm
        )) / NULLIF(LAG(avg_salary, 12) OVER (
            PARTITION BY comp_band
            ORDER BY snapshot_yyyymm
        ), 0), 2)                               AS yoy_salary_pct,

    current_timestamp()                         AS load_ts

FROM monthly_comp
ORDER BY comp_band, snapshot_yyyymm;

SELECT 'v_compensation_trend rows:' AS table_name,
       COUNT(*) AS row_count
FROM people_analytics.gold.v_compensation_trend;

In [0]:
-- =============================================================================
-- CELL 9 — Gold layer summary verification
-- =============================================================================

SELECT 'fact_headcount_snapshot'  AS gold_table,
       COUNT(*) AS rows
FROM people_analytics.gold.fact_headcount_snapshot
UNION ALL
SELECT 'fact_attrition_event', COUNT(*)
FROM people_analytics.gold.fact_attrition_event
UNION ALL
SELECT 'fact_compensation_history', COUNT(*)
FROM people_analytics.gold.fact_compensation_history
UNION ALL
SELECT 'fact_performance_review', COUNT(*)
FROM people_analytics.gold.fact_performance_review
UNION ALL
SELECT 'v_headcount_trend', COUNT(*)
FROM people_analytics.gold.v_headcount_trend
UNION ALL
SELECT 'v_attrition_trend', COUNT(*)
FROM people_analytics.gold.v_attrition_trend
UNION ALL
SELECT 'v_compensation_trend', COUNT(*)
FROM people_analytics.gold.v_compensation_trend
ORDER BY gold_table;

In [0]:
-- =============================================================================
-- CELL 10 — Headcount trend smoke test
-- Verify MoM and YoY columns have data
-- =============================================================================

SELECT
    snapshot_yyyymm,
    active_headcount,
    prior_month_hc,
    mom_hc_delta,
    mom_hc_pct,
    prior_year_hc,
    yoy_hc_pct,
    rolling_3m_avg_hc,
    rolling_12m_avg_hc
FROM people_analytics.gold.v_headcount_trend
ORDER BY snapshot_yyyymm DESC
LIMIT 15;

In [0]:
-- =============================================================================
-- CELL 11 — Attrition trend smoke test
-- =============================================================================

SELECT
    snapshot_yyyymm,
    active_hc,
    voluntary_exits,
    voluntary_attrition_rate,
    rolling_3m_attrition,
    rolling_12m_attrition,
    prior_year_attrition_rate
FROM people_analytics.gold.v_attrition_trend
ORDER BY snapshot_yyyymm DESC
LIMIT 15;

In [0]:
-- =============================================================================
-- CELL 12 — Pay equity smoke test
-- =============================================================================

SELECT
    gender,
    job_level,
    COUNT(*)                            AS headcount,
    ROUND(AVG(salary), 0)               AS avg_salary,
    ROUND(AVG(compa_ratio), 3)          AS avg_compa_ratio,
    ROUND(100.0 * COUNT(*) /
        SUM(COUNT(*)) OVER
        (PARTITION BY job_level), 1)    AS pct_of_level
FROM people_analytics.silver.dim_employee
WHERE is_current = 1
  AND status     = 'Active'
  AND gender     IN ('Male', 'Female')
GROUP BY gender, job_level
ORDER BY job_level, gender;

SELECT 'Gold layer build complete. Ready for Semantic layer.' AS status;

In [0]:
SELECT
    gender,
    job_level,
    COUNT(*)                   AS headcount,
    ROUND(AVG(salary), 0)      AS avg_salary,
    ROUND(AVG(compa_ratio), 3) AS avg_compa_ratio
FROM people_analytics.silver.dim_employee
WHERE is_current = 1
  AND status     = 'Active'
  AND gender     IN ('Male', 'Female')
  AND job_level  IN ('IC4', 'IC5', 'M1', 'M2')
GROUP BY gender, job_level
ORDER BY job_level, gender;


--This focuses specifically on IC4+ where the pay gap was engineered. Share that output and I will confirm the gap is visible before we move to Semantic.